# 25 - Time-Stratified Radiomics Analysis on Independent Dataset (Normalised)

Repeat the early-versus-late radiomics comparison on the independent dataset (55 studies, one
per patient), rather than on all 137 studies.

**Why:** The stratified analysis in NB 14a splits the full dataset by the `motivo` variable and
uses all 137 studies, which violates the independence assumption. This notebook restricts the
stratification to the independent dataset and uses actual days post-transplant, so that each
subgroup contains one independent study per patient. The 90-day cutoff matches Bassaganyas et
al. and the clinical late-period analysis, giving one consistent definition of "late" across
the thesis.

**Method (same tests as NB 14a / NB 19):**
- Split the 55 studies into early (<= 90 days) and late (> 90 days) using `Días pTXP`.
- Within each subgroup: Shapiro-Wilk normality check per feature, then Welch's t-test if both
  groups are normal, otherwise Mann-Whitney U.
- Effect size: Cohen's d (t-test) or rank-biserial correlation (Mann-Whitney).
- Benjamini-Hochberg FDR correction across the 93 features.

**Caveat:** the late subgroup is small (about a dozen studies). Results are reported honestly
with this limitation; the stratified analysis is a robustness check, not a primary result.

In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

print("Imports OK")

Imports OK


In [2]:
# load the independent dataset and attach days post-transplant from the source spreadsheet
df = pd.read_csv(os.path.join("reports", "18_independent_dataset_normalised.csv"))
df["study_id"] = df["study_id"].astype(str).str.strip()
feature_cols = [c for c in df.columns if c.startswith("original_")]

clinical = pd.read_csv(os.path.join("..", "data", "bd_estudiUPF.csv"))
clinical["id estudio"] = clinical["id estudio"].astype(str).str.strip()
days_map = dict(zip(clinical["id estudio"], clinical["Días pTXP"]))
df["days_post_tx"] = df["study_id"].map(days_map)

print(f"Independent dataset: {len(df)} studies, {len(feature_cols)} radiomics features")
print(f"Days mapped: {df['days_post_tx'].notna().sum()} / {len(df)}")

Independent dataset: 55 studies, 93 radiomics features
Days mapped: 55 / 55


In [3]:
# split into early and late using the 90-day cutoff
early = df[df["days_post_tx"] <= 90]
late = df[df["days_post_tx"] > 90]

for label, subset in [("Early (<= 90 days)", early), ("Late (> 90 days)", late)]:
    n_no = (subset["rejection"] == 0).sum()
    n_rej = (subset["rejection"] == 1).sum()
    print(f"{label}: n = {len(subset)} (no rejection {n_no}, rejection {n_rej})")

Early (<= 90 days): n = 42 (no rejection 28, rejection 14)
Late (> 90 days): n = 13 (no rejection 6, rejection 7)


## Stratified testing

The same per-feature procedure as NB 14a and NB 19 is applied within each time subgroup.

In [4]:
def run_radiomics_stats(subset, feature_cols):
    """Run per-feature Welch or Mann-Whitney tests with FDR on one subgroup."""
    rej = subset[subset["rejection"] == 1]
    no_rej = subset[subset["rejection"] == 0]

    rows = []
    for feat in feature_cols:
        a = no_rej[feat].dropna().values
        b = rej[feat].dropna().values

        _, p_a = stats.shapiro(a)
        _, p_b = stats.shapiro(b)
        both_normal = (p_a > 0.05) and (p_b > 0.05)

        if both_normal:
            stat, p_value = stats.ttest_ind(a, b, equal_var=False)
            test_name = "t-test"
            pooled_std = np.sqrt(
                ((len(a) - 1) * a.std(ddof=1) ** 2 +
                 (len(b) - 1) * b.std(ddof=1) ** 2) / (len(a) + len(b) - 2))
            effect_size = (b.mean() - a.mean()) / pooled_std if pooled_std > 0 else 0.0
        else:
            stat, p_value = stats.mannwhitneyu(a, b, alternative="two-sided")
            test_name = "Mann-Whitney"
            effect_size = 1 - (2 * stat) / (len(a) * len(b))

        rows.append({
            "feature": feat, "test": test_name, "p_value": p_value,
            "effect_size": effect_size,
            "median_no_rej": np.median(a), "median_rej": np.median(b),
        })

    out = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
    out["p_value_fdr"] = multipletests(out["p_value"].values, method="fdr_bh")[1]
    out["significant_fdr"] = out["p_value_fdr"] < 0.05
    return out

In [5]:
early_results = run_radiomics_stats(early, feature_cols)
print("EARLY (<= 90 days)")
print(f"  Uncorrected p < 0.05: {(early_results['p_value'] < 0.05).sum()} / {len(early_results)}")
print(f"  FDR-corrected p < 0.05: {early_results['significant_fdr'].sum()}")
print("  Top 5 features:")
print(early_results[["feature", "test", "p_value", "p_value_fdr", "effect_size"]].head(5).to_string(index=False))

EARLY (<= 90 days)
  Uncorrected p < 0.05: 1 / 93
  FDR-corrected p < 0.05: 0
  Top 5 features:
                              feature         test  p_value  p_value_fdr  effect_size
  original_firstorder_RootMeanSquared Mann-Whitney 0.043960      0.60733    -0.387755
 original_gldm_GrayLevelNonUniformity Mann-Whitney 0.053061      0.60733     0.372449
              original_ngtdm_Strength Mann-Whitney 0.076008      0.60733    -0.341837
     original_firstorder_90Percentile       t-test 0.091297      0.60733    -0.582440
original_gldm_DependenceNonUniformity Mann-Whitney 0.095394      0.60733     0.321429


In [6]:
late_results = run_radiomics_stats(late, feature_cols)
print("LATE (> 90 days)")
print(f"  Subgroup size: {len(late)} studies "
      f"(no rejection {(late['rejection'] == 0).sum()}, rejection {(late['rejection'] == 1).sum()})")
print(f"  Uncorrected p < 0.05: {(late_results['p_value'] < 0.05).sum()} / {len(late_results)}")
print(f"  FDR-corrected p < 0.05: {late_results['significant_fdr'].sum()}")
print("  Top 5 features:")
print(late_results[["feature", "test", "p_value", "p_value_fdr", "effect_size"]].head(5).to_string(index=False))

LATE (> 90 days)
  Subgroup size: 13 studies (no rejection 6, rejection 7)
  Uncorrected p < 0.05: 1 / 93
  FDR-corrected p < 0.05: 0
  Top 5 features:
                         feature   test  p_value  p_value_fdr  effect_size
     original_firstorder_Maximum t-test 0.038335     0.337251    -1.308614
original_glcm_MaximumProbability t-test 0.066957     0.337251     1.125098
        original_glcm_SumEntropy t-test 0.068283     0.337251    -1.180199
       original_glcm_JointEnergy t-test 0.071515     0.337251     1.126759
      original_glcm_JointEntropy t-test 0.076594     0.337251    -1.159193


In [7]:
early_path = os.path.join("reports", "25_radiomics_stratified_independent_early.csv")
late_path = os.path.join("reports", "25_radiomics_stratified_independent_late.csv")
early_results.to_csv(early_path, index=False)
late_results.to_csv(late_path, index=False)
print(f"Saved {early_path}")
print(f"Saved {late_path}")
print()
print("SUMMARY")
print(f"Early (<= 90 days): n = {len(early)}, "
      f"uncorrected p<0.05 = {(early_results['p_value'] < 0.05).sum()}, "
      f"FDR sig = {early_results['significant_fdr'].sum()}")
print(f"Late (> 90 days): n = {len(late)}, "
      f"uncorrected p<0.05 = {(late_results['p_value'] < 0.05).sum()}, "
      f"FDR sig = {late_results['significant_fdr'].sum()}")

Saved reports/25_radiomics_stratified_independent_early.csv
Saved reports/25_radiomics_stratified_independent_late.csv

SUMMARY
Early (<= 90 days): n = 42, uncorrected p<0.05 = 1, FDR sig = 0
Late (> 90 days): n = 13, uncorrected p<0.05 = 1, FDR sig = 0


## Interpretation

Neither time subgroup yields a radiomics feature that survives FDR correction. In the early
period (<= 90 days, n = 42) one feature reaches uncorrected significance; in the late period
(> 90 days, n = 13) one feature reaches uncorrected significance. In both cases the corrected
p-values are far from the 0.05 threshold, and the single uncorrected hit differs between
subgroups, indicating chance findings rather than a consistent time-dependent signal.

The late subgroup is small (13 studies, split 6 no-rejection versus 7 rejection), so it is
underpowered and its result should be read as a robustness check rather than as evidence. The
overall conclusion matches the primary independent-dataset analysis (NB 19): radiomics texture
features do not discriminate rejection, and stratifying by post-transplant period does not
reveal a hidden signal.